In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from keras.datasets import mnist

In [2]:
(X_train, y_train),(X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(-1, 784) / 255
X_test = X_test.reshape(-1, 784) / 255

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

# One hot encoding
def one_hot(Y):
    return np.eye(10)[Y]

# Initialize parameters
def init_params(input_size, hidden_size, output_size):
    W1 = np.random.randn(hidden_size, input_size) * 0.01
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(output_size, hidden_size) * 0.01
    b2 = np.zeros((1, output_size))
    return W1, b1, W2, b2

# Forward propagation
def forward_prop(W1, b1, W2, b2, X):
    Z1 = X.dot(W1.T) + b1
    A1 = ReLU(Z1)
    Z2 = A1.dot(W2.T) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

# Backward propagation
def back_prop(W1, b1, W2, b2, X, Y, A1, A2):
    m = X.shape[0]
    one_hot_Y = one_hot(Y)

    dZ2 = A2 - one_hot_Y
    dW2 = (1/m) * dZ2.T.dot(A1)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2.dot(W2)
    dZ1 = dA1 * (A1 > 0)
    dW1 = (1/m) * dZ1.T.dot(X)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2
    return W1, b1, W2, b2

In [4]:
# Training parameters
input_size = 784
hidden_size = 128
output_size = 10
epochs = 20
batch_size = 50
alpha = 0.01

In [5]:
# Initialize weights
W1, b1, W2, b2 = init_params(input_size, hidden_size, output_size)

for epoch in range(epochs):
    epoch_loss = 0

    for i in range(0, X_train.shape[0], batch_size):
        X_batch = X_train[i : i+batch_size]
        Y_batch = y_train[i : i+batch_size]

        # Forward Propagation
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X_batch)

        # Compute loss (Cross-entropy)
        one_hot_Y = one_hot(Y_batch)
        loss = -np.sum(one_hot_Y * np.log(A2)) / X_batch.shape[0]
        epoch_loss += loss

        # Backward Propagation
        dW1, db1, dW2, db2 = back_prop(W1, b1, W2, b2, X_batch, Y_batch, A1, A2)

        # Update parameters
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss / (X_train.shape[0] / batch_size)}")


Epoch 1/20, Loss: 1.623674792167317
Epoch 2/20, Loss: 0.5619600921074688
Epoch 3/20, Loss: 0.4075534243098501
Epoch 4/20, Loss: 0.3569990484133933
Epoch 5/20, Loss: 0.3296537259385506
Epoch 6/20, Loss: 0.31050486583307363
Epoch 7/20, Loss: 0.29505522199925155
Epoch 8/20, Loss: 0.2815561801266408
Epoch 9/20, Loss: 0.2692874039913993
Epoch 10/20, Loss: 0.25797430036249874
Epoch 11/20, Loss: 0.2475141398795824
Epoch 12/20, Loss: 0.2378049837539443
Epoch 13/20, Loss: 0.22874396986236795
Epoch 14/20, Loss: 0.22030488877933166
Epoch 15/20, Loss: 0.212431676392029
Epoch 16/20, Loss: 0.20504939859904514
Epoch 17/20, Loss: 0.1981256609301081
Epoch 18/20, Loss: 0.19160525454138908
Epoch 19/20, Loss: 0.18543070220507196
Epoch 20/20, Loss: 0.17960478679093123


In [6]:
def evaluate(W1, b1, W2, b2, X_test, Y_test):
    Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X_test)
    predictions = np.argmax(A2, axis=1)
    accuracy = np.mean(predictions == Y_test)
    return accuracy

accuracy = evaluate(W1, b1, W2, b2, X_test, y_test)
print(f"Test Accuracy: {accuracy * 100}%")

Test Accuracy: 94.78999999999999%
